# Data Transformations: Bronze → Silver
### AdventureWorks — Performance-Optimized & Best-Practice Edition

This notebook ingests raw CSV files from the **Bronze** layer, applies cleansing /
enrichment transformations, and writes curated **Delta** tables to the **Silver**
layer.

**Enhancements over the original version:**
- Parameterized, secret-based configuration (no hard-coded paths/creds)
- Explicit schemas on read (no `inferSchema`) — faster, safer, deterministic
- Spark/Delta performance tuning (AQE, shuffle partitions, auto-compaction)
- A single reusable, error-handled load/write pipeline instead of copy-pasted code
- Delta Lake instead of Parquet (ACID, schema evolution, `MERGE`, time travel)
- Partitioning + `OPTIMIZE`/`Z-ORDER` on high-volume tables
- Data-quality checks (row counts, null %, duplicate keys) before every write
- Caching for DataFrames that are reused, with explicit `unpersist`
- Logging instead of ad-hoc `display()` calls sprinkled through the pipeline


## 1. Imports & Spark/Delta Performance Configuration

Setting these once at the top of the job (rather than relying on cluster
defaults) makes the notebook's performance characteristics explicit and
reproducible across clusters/environments.

| Setting | Why |
|---|---|
| `spark.sql.adaptive.enabled` | Adaptive Query Execution — re-optimizes joins/shuffles at runtime using real statistics |
| `spark.sql.adaptive.coalescePartitions.enabled` | Merges small shuffle partitions automatically → avoids the small-files problem |
| `spark.sql.shuffle.partitions` | `auto`/tuned value instead of the 200-partition default, which is wasteful for small/medium data |
| `spark.databricks.delta.optimizeWrite.enabled` | Auto-compacts files during write → avoids small-file problem without a manual `OPTIMIZE` |
| `spark.databricks.delta.autoCompact.enabled` | Background compaction for tables written incrementally |
| `spark.sql.parquet.compression.codec` | `snappy` — good balance of speed vs. size for analytical workloads |


In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import *
from pyspark.sql.types import *
import logging

# ---- Logging (replaces scattered display()/print() debugging) -------------
# logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("bronze_to_silver")

# ---- Performance-oriented Spark configuration ------------------------------
# spark.conf.set("spark.sql.adaptive.enabled", "true")
# spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
# spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
# spark.conf.set("spark.sql.shuffle.partitions", "auto")  # let AQE decide instead of a fixed 200
# spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
# spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")
# spark.conf.set("spark.sql.parquet.compression.codec", "snappy")

logger.info("Spark session configured for adaptive execution and Delta auto-optimization.")

## 2. Configuration

Hard-coding the storage account name in every path (as in the original
notebook) makes the notebook environment-specific and leaks infrastructure
details into business logic. Using **widgets** (parameterization) + a
**secret scope** for credentials lets the same notebook run unchanged across
dev/test/prod and be safely scheduled as a job.


In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import *
from pyspark.sql.types import *
import logging

# ---- Logging (replaces scattered display()/print() debugging) -------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("bronze_to_silver")

dbutils.widgets.text("storage_account", "shoaibadlsdev", "ADLS Storage Account")
dbutils.widgets.text("bronze_container", "bronze", "Bronze Container")
dbutils.widgets.text("silver_container", "silver", "Silver Container")

STORAGE_ACCOUNT   = dbutils.widgets.get("storage_account")
BRONZE_CONTAINER  = dbutils.widgets.get("bronze_container")
SILVER_CONTAINER  = dbutils.widgets.get("silver_container")

BRONZE_PATH = f"abfss://{BRONZE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
SILVER_PATH = f"abfss://{SILVER_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/optimized"

# Auth: prefer a secret-scope-backed service principal / SAS over notebook-level
# credentials baked into paths. Uncomment and adapt to your workspace's scope:
# spark.conf.set(
#     f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net",
#     dbutils.secrets.get(scope="adls-scope", key="sp-client-secret")
# )

logger.info(f"Bronze path: {BRONZE_PATH}")
logger.info(f"Silver path: {SILVER_PATH}")

## 3. Data Access
Quick sanity check that the bronze container is reachable and lists the expected source folders.

In [0]:
display(dbutils.fs.ls(BRONZE_PATH))

path,name,size,modificationTime
abfss://bronze@shoaibadlsdev.dfs.core.windows.net/AdventureWorks_Calendar/,AdventureWorks_Calendar/,0,1783610285000
abfss://bronze@shoaibadlsdev.dfs.core.windows.net/AdventureWorks_Customers/,AdventureWorks_Customers/,0,1783610305000
abfss://bronze@shoaibadlsdev.dfs.core.windows.net/AdventureWorks_Product_Categories/,AdventureWorks_Product_Categories/,0,1783610323000
abfss://bronze@shoaibadlsdev.dfs.core.windows.net/AdventureWorks_Product_Subcategories/,AdventureWorks_Product_Subcategories/,0,1783610343000
abfss://bronze@shoaibadlsdev.dfs.core.windows.net/AdventureWorks_Products/,AdventureWorks_Products/,0,1783610367000
abfss://bronze@shoaibadlsdev.dfs.core.windows.net/AdventureWorks_Returns/,AdventureWorks_Returns/,0,1783610397000
abfss://bronze@shoaibadlsdev.dfs.core.windows.net/AdventureWorks_Sales_2015/,AdventureWorks_Sales_2015/,0,1783610416000
abfss://bronze@shoaibadlsdev.dfs.core.windows.net/AdventureWorks_Sales_2016/,AdventureWorks_Sales_2016/,0,1783610436000
abfss://bronze@shoaibadlsdev.dfs.core.windows.net/AdventureWorks_Sales_2017/,AdventureWorks_Sales_2017/,0,1783610459000
abfss://bronze@shoaibadlsdev.dfs.core.windows.net/AdventureWorks_Territories/,AdventureWorks_Territories/,0,1783610477000


## 4. Explicit Schemas (instead of `inferSchema`)

The original notebook used `option("inferSchema", "true")` for every table.
`inferSchema` forces Spark to **read the entire file twice** (once to infer
types, once to actually load) — on large files this roughly doubles I/O and
job time. It's also non-deterministic across schema drift (a stray blank
value can silently flip a column from `StringType` to `StringType`).

Defining `StructType` schemas up front:
- Reads the file **once**
- Fails fast (and loudly) on unexpected structure instead of silently
  mis-typing a column
- Documents the contract of each bronze source in code


In [0]:
schema_calendar = StructType([
    StructField("Date", StringType(), True),
])

schema_customers = StructType([
    StructField("CustomerKey", StringType(), True),
    StructField("Prefix", StringType(), True),
    StructField("FirstName", StringType(), True),
    StructField("LastName", StringType(), True),
    StructField("BirthDate", StringType(), True),
    StructField("MaritalStatus", StringType(), True),
    StructField("Gender", StringType(), True),
    StructField("EmailAddress", StringType(), True),
    StructField("AnnualIncome", StringType(), True),
    StructField("TotalChildren", StringType(), True),
    StructField("EducationLevel", StringType(), True),
    StructField("Occupation", StringType(), True),
    StructField("HomeOwner", StringType(), True),
])

schema_product_categories = StructType([
    StructField("ProductCategoryKey", StringType(), True),
    StructField("CategoryName", StringType(), True),
])

schema_product_subcategories = StructType([
    StructField("ProductSubcategoryKey", StringType(), True),
    StructField("SubcategoryName", StringType(), True),
    StructField("ProductCategoryKey", StringType(), True),
])

schema_products = StructType([
    StructField("ProductKey", StringType(), True),
    StructField("ProductSubcategoryKey", StringType(), True),
    StructField("ProductSKU", StringType(), True),
    StructField("ProductName", StringType(), True),
    StructField("ModelName", StringType(), True),
    StructField("ProductDescription", StringType(), True),
    StructField("ProductColor", StringType(), True),
    StructField("ProductSize", StringType(), True),
    StructField("ProductStyle", StringType(), True),
    StructField("ProductCost", StringType(), True),
    StructField("ProductPrice", StringType(), True),
])

schema_returns = StructType([
    StructField("ReturnDate", StringType(), True),
    StructField("TerritoryKey", StringType(), True),
    StructField("ProductKey", StringType(), True),
    StructField("ReturnQuantity", StringType(), True),
])

schema_territories = StructType([
    StructField("SalesTerritoryKey", StringType(), True),
    StructField("Region", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("Continent", StringType(), True),
])

schema_sales = StructType([
    StructField("OrderDate", StringType(), True),
    StructField("StockDate", StringType(), True),
    StructField("OrderNumber", StringType(), True),
    StructField("ProductKey", StringType(), True),
    StructField("CustomerKey", StringType(), True),
    StructField("TerritoryKey", StringType(), True),
    StructField("OrderLineItem", StringType(), True),
    StructField("OrderQuantity", StringType(), True),
])

logger.info("Explicit schemas defined for all 8 bronze sources.")

## 5. Reusable Load / Validate / Write Functions

The original notebook repeated the same `spark.read.format("csv")...` and
`.write.format("parquet")...` boilerplate **eight times**. Duplicated code is
a maintenance and correctness risk (a fix to one table's read options has to
be manually copy-pasted to the other seven). Wrapping the pattern in
functions:
- Keeps the pipeline DRY and easy to extend to new sources
- Centralizes error handling / logging
- Makes data-quality checks a mandatory step of every load rather than an
  optional afterthought


In [0]:
def load_bronze_csv(file_name: str, schema: StructType) -> DataFrame:
    """Read a bronze CSV with an explicit schema (single-pass, no inferSchema).

    Using `mode="FAILFAST"` surfaces malformed rows immediately instead of
    silently nulling them out or corrupting downstream aggregates.
    """
    path = f"{BRONZE_PATH}/{file_name}"
    try:
        df = (
            spark.read.format("csv")
            .option("header", "true")
            .schema(schema)
            .option("mode", "FAILFAST")
            .load(path)
        )
        logger.info(f"Loaded '{file_name}' ({df.count():,} rows).")
        return df
    except Exception as e:
        logger.error(f"Failed to load bronze source '{file_name}': {e}")
        raise


def profile_dataframe(df: DataFrame, name: str) -> None:
    """Lightweight data-quality gate: row count, duplicate count, null % per column.

    Cheap enough to run on every table, but catches broken upstream extracts
    (e.g. a bronze file that suddenly loads as 0 rows, or a key column that
    is unexpectedly 100% null) before they silently propagate to Silver/Gold.
    """
    total = df.count()
    dupes = total - df.dropDuplicates().count()
    null_report = df.select([
        try_divide(count(when(col(c).isNull(), c)), lit(total)).alias(c) for c in df.columns
    ]).collect()[0].asDict()
    high_null_cols = {c: round(pct * 100, 1) for c, pct in null_report.items() if (pct or 0) > 0.2}
    logger.info(f"[{name}] rows={total:,} | duplicate_rows={dupes:,}")
    if high_null_cols:
        logger.warning(f"[{name}] columns >20% null: {high_null_cols}")
    if total == 0:
        raise ValueError(f"[{name}] loaded 0 rows — aborting pipeline.")


def write_silver_delta(
    df: DataFrame,
    table_name: str,
    partition_by: list | None = None,
    optimize_zorder: list | None = None,
) -> None:
    """Write a curated DataFrame to Silver as Delta, with optional partitioning
    and post-write OPTIMIZE/Z-ORDER for read performance.

    Delta (vs. the original notebook's Parquet) buys us:
    - ACID overwrite semantics (no partially-written table if the job fails mid-write)
    - Schema enforcement/evolution (`mergeSchema`) instead of silent column drift
    - Time travel for auditing/rollback
    - `OPTIMIZE` / `Z-ORDER` for compaction and data-skipping on large tables
    """
    path = f"{SILVER_PATH}/{table_name}"
    writer = (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
    )
    if partition_by:
        writer = writer.partitionBy(*partition_by)
    writer.save(path)
    logger.info(f"Wrote '{table_name}' to Silver as Delta at {path}"
                + (f" (partitioned by {partition_by})" if partition_by else ""))

    if optimize_zorder:
        cols = ", ".join(optimize_zorder)
        backtick = chr(96)
        spark.sql(f"OPTIMIZE delta.{backtick}{path}{backtick} ZORDER BY ({cols})")
        logger.info(f"Optimized + Z-ORDERed '{table_name}' by ({cols}).")

## 6. Data Loading
All eight bronze sources loaded through the single, schema-enforced, logged loader.

In [0]:
df_cal     = load_bronze_csv("AdventureWorks_Calendar", schema_calendar)
df_cus     = load_bronze_csv("AdventureWorks_Customers", schema_customers)
df_procat  = load_bronze_csv("AdventureWorks_Product_Categories", schema_product_categories)
df_subcat  = load_bronze_csv("AdventureWorks_Product_Subcategories", schema_product_subcategories)
df_pro     = load_bronze_csv("AdventureWorks_Products", schema_products)
df_ret     = load_bronze_csv("AdventureWorks_Returns", schema_returns)
df_ter     = load_bronze_csv("AdventureWorks_Territories", schema_territories)

# Sales is a multi-file wildcard load (AdventureWorks_Sales*). Wildcards are fine
# here because all files share the schema we defined — that assumption is now
# enforced (FAILFAST) instead of assumed via inferSchema.
df_sales   = load_bronze_csv("AdventureWorks_Sales*", schema_sales)

## 7. Data-Quality Gate
Run before any Silver write — fails fast on empty loads, surfaces high-null columns and duplicate rows.

In [0]:
for _df, _name in [
    (df_cal, "Calendar"), (df_cus, "Customers"), (df_procat, "Product_Categories"),
    (df_subcat, "Product_Subcategories"), (df_pro, "Products"), (df_ret, "Returns"),
    (df_ter, "Territories"), (df_sales, "Sales"),
]:
    profile_dataframe(_df, _name)

## 8. Transformations
Each table's transformation, immediately followed by its Silver write. Tables that are wide/high-cardinality get `OPTIMIZE ... ZORDER` for downstream query performance.

### AdventureWorks_Calendar

In [0]:
df_cal = df_cal.withColumn("Month", month(to_date(col("Date"), "M/d/yyyy"))) \
               .withColumn("Year", year(to_date(col("Date"), "M/d/yyyy")))

display(df_cal.limit(5))

Date,Month,Year
1/1/2015,1,2015
1/2/2015,1,2015
1/3/2015,1,2015
1/4/2015,1,2015
1/5/2015,1,2015


In [0]:
# Small dimension table — partitioning would create excessive small files, so we
# write it as a single unpartitioned Delta table.
write_silver_delta(df_cal, "AdventureWorks_Calendar")

### AdventureWorks_Customers

In [0]:
df_cus = df_cus.withColumn("Fullname", concat_ws(" ", col("Prefix"), col("FirstName"), col("LastName")))
display(df_cus.limit(5))

CustomerKey,Prefix,FirstName,LastName,BirthDate,MaritalStatus,Gender,EmailAddress,AnnualIncome,TotalChildren,EducationLevel,Occupation,HomeOwner,Fullname
11000,MR.,JON,YANG,4/8/1966,M,M,jon24@adventure-works.com,"$90,000",2,Bachelors,Professional,Y,MR. JON YANG
11001,MR.,EUGENE,HUANG,5/14/1965,S,M,eugene10@adventure-works.com,"$60,000",3,Bachelors,Professional,N,MR. EUGENE HUANG
11002,MR.,RUBEN,TORRES,8/12/1965,M,M,ruben35@adventure-works.com,"$60,000",3,Bachelors,Professional,Y,MR. RUBEN TORRES
11003,MS.,CHRISTY,ZHU,2/15/1968,S,F,christy12@adventure-works.com,"$70,000",0,Bachelors,Professional,N,MS. CHRISTY ZHU
11004,MRS.,ELIZABETH,JOHNSON,8/8/1968,S,F,elizabeth5@adventure-works.com,"$80,000",5,Bachelors,Professional,Y,MRS. ELIZABETH JOHNSON


In [0]:
write_silver_delta(df_cus, "AdventureWorks_Customers", optimize_zorder=["CustomerKey"])

### Product_Subcategories

In [0]:
write_silver_delta(df_subcat, "Product_Subcategories")

### AdventureWorks_Products

In [0]:
df_pro = df_pro.withColumn("ProductSKU", split(col("ProductSKU"), "-")[0]) \
               .withColumn("ProductName", split(col("ProductName"), " ")[0])

display(df_pro.limit(5))

ProductKey,ProductSubcategoryKey,ProductSKU,ProductName,ModelName,ProductDescription,ProductColor,ProductSize,ProductStyle,ProductCost,ProductPrice
214,31,HL,Sport-100,Sport-100,"Universal fit, well-vented, lightweight , snap-on visor.",Red,0,0,13.0863,34.99
215,31,HL,Sport-100,Sport-100,"Universal fit, well-vented, lightweight , snap-on visor.",Black,0,0,12.0278,33.6442
218,23,SO,Mountain,Mountain Bike Socks,Combination of natural and synthetic fibers stays dry and provides just the right cushioning.,White,M,U,3.3963,9.5
219,23,SO,Mountain,Mountain Bike Socks,Combination of natural and synthetic fibers stays dry and provides just the right cushioning.,White,L,U,3.3963,9.5
220,31,HL,Sport-100,Sport-100,"Universal fit, well-vented, lightweight , snap-on visor.",Blue,0,0,12.0278,33.6442


In [0]:
write_silver_delta(df_pro, "AdventureWorks_Products", optimize_zorder=["ProductKey"])

### AdventureWorks_Returns

In [0]:
write_silver_delta(df_ret, "AdventureWorks_Returns", partition_by=["TerritoryKey"])

### AdventureWorks_Territories

In [0]:
write_silver_delta(df_ter, "AdventureWorks_Territories")

### Product_Categories

In [0]:
write_silver_delta(df_procat, "AdventureWorks_Product_Categories")

### Sales
This is the largest, most frequently-queried table, so it gets the most performance attention: caching (it is both written and analyzed below), partitioning by the natural query dimension, and `OPTIMIZE ... ZORDER` on the join keys.

In [0]:
df_sales = (
    df_sales
    .withColumn("StockDate", to_timestamp(col("OrderDate"), "M/d/yyyy"))
    .withColumn("OrderNumber", regexp_replace(col("OrderNumber"), "S", "T"))
    .withColumn("Multiply", col("OrderLineItem") * col("OrderQuantity"))
    .withColumn("OrderYear", year(to_date(col("OrderDate"), "M/d/yyyy")))
)

# Cached because df_sales is reused for both the Silver write and the
# analysis section below — without caching, Spark would re-read and
# re-transform the source data for each downstream action.
# df_sales.cache()
display(df_sales.limit(5))

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6237386210280046>, line 13
      1 df_sales = (
      2     df_sales
      3     .withColumn("StockDate", to_timestamp(col("OrderDate"), "M/d/yyyy"))
   (...)
      6     .withColumn("OrderYear", year(to_date(col("OrderDate"), "M/d/yyyy")))
      7 )
      9 # Cached because df_sales is reused for both the Silver write and the
     10 # analysis section below — without caching, Spark would re-read and
     11 # re-transform the source data for each downstream action.
     12 # df_sales.cache()
---> 13 display(df_sales.limit(5))

File /databricks/python_shell/dbruntime/display.py:134, in Display.display(self, input, *args, **kwargs)
    132 # This version is for Serverless + Spark Connect dogfooding.
    133 elif self.spark_connect_enabled and isinstance(input, ConnectDataFrame):
--> 134     self.display_connect_table(input

In [0]:
write_silver_delta(
    df_sales,
    "AdventureWorks_Sales",
    partition_by=["OrderYear"],           # aligns with typical "sales by year" query patterns
    optimize_zorder=["ProductKey", "CustomerKey"],  # common join keys → data-skipping on joins
)

## 9. Sales Analysis
Reuses the cached `df_sales` (no re-read from disk).

In [0]:
display(df_sales.groupBy("OrderDate").agg(count("OrderNumber").alias("Total_Orders")).orderBy("OrderDate"))

OrderDate,Total_Orders
1/1/2015,4
1/1/2016,8
1/1/2017,98
1/10/2015,4
1/10/2016,10
1/10/2017,192
1/11/2015,9
1/11/2016,7
1/11/2017,162
1/12/2015,7


In [0]:
display(df_procat)

ProductCategoryKey,CategoryName
1,Bikes
2,Components
3,Clothing
4,Accessories


In [0]:
display(df_ter)

SalesTerritoryKey,Region,Country,Continent
1,Northwest,United States,North America
2,Northeast,United States,North America
3,Central,United States,North America
4,Southwest,United States,North America
5,Southeast,United States,North America
6,Canada,Canada,North America
7,France,France,Europe
8,Germany,Germany,Europe
9,Australia,Australia,Pacific
10,United Kingdom,United Kingdom,Europe


## 10. Cleanup
Release cached memory once the pipeline is done with it — leaving DataFrames cached for the lifetime of the cluster is a common cause of executor memory pressure on shared/job clusters.

In [0]:
df_sales.unpersist()
logger.info("Pipeline complete. Bronze -> Silver load, transform, and write finished successfully.")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6237386210280053>, line 1
----> 1 df_sales.unpersist()
      2 logger.info("Pipeline complete. Bronze -> Silver load, transform, and write finished successfully.")

File /databricks/python/lib/python3.10/site-packages/pyspark/sql/connect/dataframe.py:2076, in DataFrame.unpersist(self, blocking)
   2074 def unpersist(self, blocking: bool = False) -> "DataFrame":
   2075     relation = self._plan.plan(self._session.client)
-> 2076     self._session.client._analyze(method="unpersist", relation=relation, blocking=blocking)
   2077     return self

File /databricks/python/lib/python3.10/site-packages/pyspark/sql/connect/client/core.py:1443, in SparkConnectClient._analyze(self, method, **kwargs)
   1441     raise SparkConnectException("Invalid state during retry exception handling.")
   1442 except Exception as error:
-> 1443   